<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/lab-11-capstone-starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone starter — an end-to-end deep learning project
**Course 1: Hands-On Deep Learning with Python — Chapter 11: Capstone**

This is a **scaffold**, not a solved lab — the capstone is your own scenario. Pick one cast
scenario from Chapters 1–9 (or propose your own with a real, public dataset), then fill in
every section below. The structure mirrors the five required deliverables exactly, so a
complete notebook here *is* most of the capstone submission.

**Constraints:** real, public dataset · runs on the Colab free tier (with an offline
fallback path if a download fails) · your own code beyond the shared `Trainer`-style
scaffold below.

## 1. Problem statement + scoping note
- **Cast scenario / org:** _fill in_
- **The business/research problem, in the stakeholder's words:** _fill in_
- **Is deep learning the right tool here — why or why not?** _fill in_
- **Target metric(s) and the threshold that counts as success:** _fill in_
- **Dataset (name, source URL, license):** _fill in_

## 2. Reproducible training pipeline

In [ ]:
!pip install -q mlflow

import numpy as np
import torch
import torch.nn as nn
import mlflow

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
mlflow.set_experiment('capstone')

In [ ]:
def load_data():
    """TODO: load your real dataset here, with a try/except offline fallback that generates
    a small synthetic stand-in of the same shape — the pattern every lab in this course used.
    Return train/val splits, properly separated (temporal split for time series, stratified
    for classification — whichever is correct for your task)."""
    raise NotImplementedError('Fill this in for your capstone scenario.')

# X_train, y_train, X_val, y_val = load_data()

In [ ]:
class Trainer:
    """A small reusable scaffold in the spirit of every lab you've built so far — fill in
    `model`, `loss_fn`, and the metric(s) that matter for your task."""
    def __init__(self, model, loss_fn, lr=1e-3, weight_decay=0.0):
        self.model = model.to(device)
        self.loss_fn = loss_fn
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    def fit(self, train_loader, val_loader, n_epochs, run_name='run'):
        history = {'train_loss': [], 'val_loss': []}
        input_example = None
        with mlflow.start_run(run_name=run_name):
            mlflow.log_params({'lr': self.optimizer.param_groups[0]['lr'], 'n_epochs': n_epochs})
            for epoch in range(n_epochs):
                self.model.train()
                epoch_loss, nb = 0.0, 0
                for xb, yb in train_loader:
                    if input_example is None:
                        input_example = xb[:1].cpu().numpy()  # mlflow requires one, as of mlflow 3.x
                    xb, yb = xb.to(device), yb.to(device)
                    self.optimizer.zero_grad()
                    loss = self.loss_fn(self.model(xb), yb)
                    loss.backward()
                    self.optimizer.step()
                    epoch_loss += loss.item(); nb += 1
                history['train_loss'].append(epoch_loss / nb)

                self.model.eval()
                val_loss, nvb = 0.0, 0
                with torch.no_grad():
                    for xb, yb in val_loader:
                        xb, yb = xb.to(device), yb.to(device)
                        val_loss += self.loss_fn(self.model(xb), yb).item(); nvb += 1
                history['val_loss'].append(val_loss / nvb)
                mlflow.log_metrics({'train_loss': history['train_loss'][-1],
                                     'val_loss': history['val_loss'][-1]}, step=epoch)
            mlflow.pytorch.log_model(self.model, 'model', input_example=input_example, serialization_format='pickle')
        return history

## 3. Evaluation scorecard
Baseline + at least one ablation + an error analysis, as the deliverable requires.

In [ ]:
# TODO:
# 1. A naive baseline (majority class / moving average / linear model — whatever is
#    appropriate) for comparison.
# 2. Your trained model's metric(s) against that baseline.
# 3. At least one ablation (e.g. "with vs without X") with a measured effect.
# 4. An error analysis: where does the model fail, and what do the failures have in common?

## 4. Deploy + monitor
Wrap your trained model in a `predict()`-style service (Chapter 10's pattern), and implement
at least one real drift check on a shifted or time-split holdout — a from-scratch PSI/KS
check is enough; reuse the Lab 10 pattern if it fits your data type.

In [ ]:
class ModelService:
    def __init__(self, model):
        self.model = model
        self.log = []

    def predict(self, x):
        # TODO: validate input, run the model, return a structured result, append to self.log
        raise NotImplementedError

    def health(self):
        return {'status': 'ok', 'requests_served': len(self.log)}

# TODO: feed a "clean" holdout, then a deliberately shifted one, and check a drift metric
# (PSI, KS, or an embedding-distance check — whichever fits your data) actually fires.

## 5. Decision dossier (2–3 pages, fill in)
- **What was built:** _fill in_
- **Cost / latency:** _fill in_
- **Risks and failure modes:** _fill in_
- **Monitoring / retraining recommendation:** _fill in_
- **Verdict — ship / don't ship / ship with conditions:** _fill in, with justification_

---
*Beacon AI · AIBits Academy — Chapter 11: Capstone*